<h1 style="color:lightblue; text-align:center;">Datasci 151 Final Project</h1>
<h3 style="color:lightblue; text-align:center;">Neko, Tae, William</h3>

In [93]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
pd.options.mode.chained_assignment = None

<h2 style="color:lightblue; text-align:center;">Preliminary Data Description</h2>
First, given that the project required data cleaning, numerical calculation, and plotting, we imported three libraries - pandas, numpy, and matplotlib.pyplot.

We used four data throughout the analysis - results.csv, constructors.csv, drivers.csv, and pit_stops.csv.

results.csv: The data has 25660 rows, where each observational unit is a specific driver's performance in a specific race. The variables we are interested in are points, millisecond, and rank. Points here means the numerical points awarded to the driver based on their finishing position. We would use this as our response variable in some analysis. Millisecond in this dataset means the race duration a driver takes. The units was changed to minutes in subsequent data analysis. This is another measure for the drivers' performance. Points demonstrates a driver's comparative performance, while the race duration shows a driver's absolute performance. We provide both for subsequent data analysis. Rank means the order of the driver's fastest lap compared to other drivers in the same race. Given the bonus point system in F1, we want to see if rank have any effect on points.

constructors.csv: The data has 211 rows, where each observational unit is a specific constructor. The variable we are interested in is the constructor's nationality. There's always debate on which country produces the best cars. Thus, we want to see if constructor's nationality have any effect on points. 

drivers.csv: The data has 854 rows, where each observational unit is a specific driver/racer. The variable we are interested in are the driver's nationality and date of birth. We want to see which countries have the most drivers or better driver performance. The date of birth could be a variable or controlled factor. It might be the case that older drivers were more experienced. However young drivers may have more enhanced race cars and better equipments.

pit_stops.csv: The data has 9299 rows, where each observational unit is a specific driver in a specific races' specific stop. The variable we are interested in is duration, which is the pit stop time. Note some calculated the pit stop time to be only the time for changing tire, which should be about 2 to 3 seconds on average. This dataset used another definition of pit stop time that also included pit stop duration and pit lane entry/exit time. We want to see if pit stop time have any effect on drivers' points.

races.csv: The data has 1079 rows, where each observational unit is a specific race. The variable we are interested in is year, the time the race took place in. Again, this variable is needed for controlling any biases given that drivers' performances  in different year segments weren't apple to apple comparisons. It should be strongly related with the race duration given technological improvements.



In [ ]:
df_results = pd.read_csv("1-Formula_One/results.csv")
df_constructors = pd.read_csv("1-Formula_One/constructors.csv")
df_drivers = pd.read_csv("1-Formula_One/drivers.csv")
df_pitstops = pd.read_csv("1-Formula_One/pit_stops.csv")
df_races = pd.read_csv("1-Formula_One/races.csv")
display(len(df_results))
display(len(df_constructors))
display(len(df_drivers))
display(len(df_pitstops))
display(len(df_races))
mean_pit_stop_time=(df_pitstops_exclude_outlier["milliseconds"].mean())/1000




25660

211

854

9299

1079

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25655,25661,1086,825,210,20,13,16,16,16,0.0,69,\N,\N,37,15,1:23.511,188.856,11
25656,25662,1086,848,3,23,17,17,17,17,0.0,69,\N,\N,43,12,1:23.047,189.911,11
25657,25663,1086,849,3,6,19,18,18,18,0.0,69,\N,\N,60,8,1:22.478,191.221,11
25658,25664,1086,852,213,22,16,19,19,19,0.0,68,\N,\N,58,16,1:23.538,188.795,12


24.33223463562073


<h2 style="color:lightblue; text-align:center;">Data Cleaning</h2>

We first converted several quantitative variables to numeric format by replacing all non-numeric values with np.nan. Then, we renamed some variables to avoid confusion; for example, we clarified the difference between "constructor_nationality" and "driver_nationality."

Major adjustments were made to the pitstops.csv dataset. We noticed that several pit stop times were extremely large, which skewed the average value significantly. These instances likely represent rare situations where the racing car was severely damaged and did not finish the race. Since these are exceptional cases, we decided to remove them so that they would not disproportionately affect the mean values.

We also refined the time variables. For time-related fields, we converted them to datetime format to enable correct plotting of graphs and further time-series analysis. For the year variable in race.csv, we recoded it into categorical variables to better suit our data analysis objectives.

In [95]:

df_results = df_results[["resultId","raceId","driverId","constructorId","points","rank","milliseconds"]]
df_results["race_time_ms"] = pd.to_numeric(df_results["milliseconds"], errors = "coerce")
df_results['race_duration_minutes'] = df_results['race_time_ms'] / (1000*60)
df_results["rank"] = pd.to_numeric(df_results["rank"], errors='coerce')
df_results["points"] = pd.to_numeric(df_results["points"], errors='coerce')
df_constructors.rename(columns={'nationality':'constructor_nationality'}, inplace=True)
df_drivers.rename(columns={'nationality':'driver_nationality'}, inplace=True)
df_drivers['date_of_birth'] = pd.to_datetime(df_drivers['dob'], format='%Y-%m-%d')
df_pitstops_exclude_outlier=df_pitstops.query("milliseconds<50000")
df_pitstops_groupby = (df_pitstops_exclude_outlier.groupby(["driverId"])
                        .agg(mean_pit_stop_time = ('milliseconds','mean')).reset_index())
df_pitstops_groupby["mean_pit_stop_time"] = np.round(df_pitstops_groupby["mean_pit_stop_time"]/100,2)

lat_bins = [1950, 1960, 1970,1980,1990,2000,2010,2023]
lat_labels = ["1950-1960", "1960-1970","1970-1980","1980-1990","1990-2000","2000-2010","2010-2022"]
df_races["year_segment"] = pd.cut(df_races["year"], 
                                bins = lat_bins,
                              right = False,
                               labels = lat_labels)


<h2 style="color:lightblue; text-align:center;">Data Cleaning</h2>

Lastly, we merged all five datasets based on the *Id shared by both data through each round of merge. 

Result.csv is our main dataset. We appended constructors.csv, drivers.csv, race.csv, and pitstops.csv in order. 

In [96]:
merge_1 = pd.merge(left = df_results,
                       right = df_constructors[["constructorId", "constructor_nationality"]],
                       on = "constructorId",
                       how = "left")
merge_2 = pd.merge(left = merge_1,
                       right = df_drivers[["driverId", "driver_nationality","date_of_birth"]],
                       on = "driverId",
                       how = "left")

merge_3 = pd.merge(left = merge_2,
                   right = df_races[["raceId","year_segment"]],
                   on = "raceId",
                   how = "left")
final_merge = pd.merge(left = merge_3,
                       right = df_pitstops_groupby[["driverId", "mean_pit_stop_time"]],
                       on = "driverId",
                       how = "left")


<h2 style="color:lightblue; text-align:center;"> Main Dataset after cleaning</h2>
Lastly, we grouped the data by driver and constructor, and calculated relevant statistics for each combination. We removed any rows where the standard deviation of race duration was zero (indicating only one observational unit) or where there were missing values, in order to avoid bias in our results.

In addition to using the categorical variables described in the data documentation, we created several descriptive quantitative variables: the mean and standard deviation for points, the mean and standard deviation for race duration (in minutes), and the mean pit stop time (in seconds). These variables serve as references and will be utilized in further stages of our data analysis.

In [97]:
result_groupby = (final_merge.groupby(["driverId","constructorId"])
                        .agg(mean_rank = ('rank','mean'),
                             sd_rank = ('rank','std'),
                             mean_race_duration_minutes = ('race_duration_minutes','mean'),
                             sd_race_duration_minutes = ('race_duration_minutes','std'),
                             mean_points = ('points','mean'),
                             sd_points = ('points','std'),
                             driver_nationality = ('driver_nationality', 'first'),
                      constructor_nationality = ('constructor_nationality', 'first'),
                      mean_pit_stop_time_second= ('mean_pit_stop_time','first'),
                      year_segment = ('year_segment','first'),
                      date_of_birth = ('date_of_birth', 'first'),).reset_index())

result_groupby["specific_driver_constructor"] = (
    result_groupby["driverId"].astype(str) + " & " + 
    result_groupby["constructorId"].astype(str)
)
final_result = result_groupby.query('sd_race_duration_minutes!=0 & sd_race_duration_minutes.notna()')
display(final_result)


,driverId,constructorId,mean_rank,sd_rank,mean_race_duration_minutes,sd_race_duration_minutes,mean_points,sd_points,driver_nationality,constructor_nationality,mean_pit_stop_time_second,year_segment,date_of_birth,specific_driver_constructor
0,1,1,6.046296,5.226432,97.280427,14.860624,8.300000,7.566888,British,British,234.82,2000-2010,1985-01-07,1 & 1
1,1,131,3.759162,3.380222,97.021280,16.764644,17.777487,8.316374,British,German,234.82,2010-2022,1985-01-07,1 & 131
2,2,2,8.318841,4.258185,93.419844,10.735130,2.328571,2.512124,German,German,229.33,2000-2010,1977-05-10,2 & 2
3,2,3,9.615385,4.292211,95.419007,6.639977,2.000000,3.113247,German,British,229.33,2000-2010,1977-05-10,2 & 3
4,2,4,12.818182,6.615409,94.379271,4.369270,3.090909,4.526689,German,French,229.33,2010-2022,1977-05-10,2 & 4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2125,849,3,15.057692,4.569350,97.518248,31.864266,0.134615,0.840841,Canadian,British,253.60,2010-2022,1995-06-29,849 & 3
2128,852,213,9.885714,6.023344,101.427097,36.974422,1.228571,2.734160,Japanese,Italian,251.50,2010-2022,2000-05-11,852 & 213
2129,853,210,15.318182,7.504832,68.914908,91.825605,0.000000,0.000000,Russian,American,255.23,2010-2022,1999-03-02,853 & 210
2130,854,210,13.485714,5.710524,91.971940,44.293469,0.342857,1.493965,German,American,259.85,2010-2022,1999-03-22,854 & 210
